In [1]:
import xrfclk
import xrfdc
import pynq
from pynq import Overlay, MMIO
from pynq import lib
import numpy as np
import time
import os
import subprocess
import time 

In [2]:
CLOCKWIZARD_LOCK_ADDRESS = 0x0004
CLOCKWIZARD_RESET_ADDRESS = 0x0000
CLOCKWIZARD_RESET_TOKEN = 0x000A
MTS_START_TILE = 0x01
MAX_DAC_TILES = 4
MAX_ADC_TILES = 4
DAC_REF_TILE = 2
ADC_REF_TILE = 2

RFSOC4X2_LMK_FREQ = 500.0
RFSOC4X2_LMX_FREQ = 500.0
RFSOC4X2_DAC_TILES = 0b0101
RFSOC4X2_ADC_TILES = 0b0101

In [3]:
xrfclk.set_ref_clks(lmk_freq = RFSOC4X2_LMK_FREQ, lmx_freq = RFSOC4X2_LMX_FREQ)
#xrfclk.set_ref_clks(lmk_freq = 245.76, lmx_freq = 491.52)
#xrfclk.set_ref_clks(lmk_freq = 245.76, lmx_freq = 409.6)

In [4]:
board = os.getenv('BOARD') 
# Run lsmod command to get the loaded modules list
output = subprocess.check_output(['lsmod'])
# Check if "zocl" is present in the output
if b'zocl' in output:
    # If present, remove the module using rmmod command
    rmmod_output = subprocess.run(['rmmod', 'zocl'])
    # Check return code
    assert rmmod_output.returncode == 0, "Could not restart zocl. Please Shutdown All Kernels and then restart"
    # If successful, load the module using modprobe command
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    assert modprobe_output.returncode == 0, "Could not restart zocl. It did not restart as expected"
else:
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    # Check return code
    assert modprobe_output.returncode == 0, "Could not restart ZOCL!"

In [5]:
#ol = Overlay('/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/Tx_solo/DIS.bit',ignore_version=True)
ol = Overlay('/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/Tx_solo_PRBS/DIS.bit',ignore_version=True)

In [6]:
#xrfclk.set_ref_clks(lmk_freq = RFSOC4X2_LMK_FREQ, lmx_freq = RFSOC4X2_LMX_FREQ)
ol.ACTIVE_DAC_TILES = RFSOC4X2_DAC_TILES
ol.ACTIVE_ADC_TILES = RFSOC4X2_ADC_TILES

In [7]:
ol.xrfdc = ol.usp_rf_data_converter_0

In [8]:
ol.xrfdc.mts_dac_config.RefTile = DAC_REF_TILE  # DAC tile distributing reference clock
ol.xrfdc.mts_adc_config.RefTile = ADC_REF_TILE  # ADC 

In [9]:
#INIT SYNC TILES

In [10]:
ol.xrfdc.mts_dac_config.Tiles = 0b0001 # turn only one tile on first
ol.xrfdc.mts_adc_config.Tiles = 0b0001
ol.xrfdc.mts_dac_config.SysRef_Enable = 1
ol.xrfdc.mts_adc_config.SysRef_Enable = 1
ol.xrfdc.mts_dac_config.Target_Latency = -1
ol.xrfdc.mts_adc_config.Target_Latency = -1

In [11]:
ol.xrfdc.mts_dac()
ol.xrfdc.mts_adc()

In [12]:
ol.Cloclk_Tree.MTS_clkwiz.mmio.write_reg(CLOCKWIZARD_RESET_ADDRESS, CLOCKWIZARD_RESET_TOKEN)

In [13]:
time.sleep(0.1)
# Reset only user selected DAC tiles
bitvector = ol.ACTIVE_DAC_TILES
for n in range(MAX_DAC_TILES):
    if (bitvector & 0x1):
        ol.xrfdc.dac_tiles[n].Reset()
    bitvector = bitvector >> 1
# Reset ADC FIFO of only user selected tiles - restarts MTS engine
for toggleValue in range(0,1):
    bitvector = ol.ACTIVE_ADC_TILES
    for n in range(MAX_ADC_TILES):
        if (bitvector & 0x1):
            ol.xrfdc.adc_tiles[n].SetupFIFOBoth(toggleValue)
        bitvector = bitvector >> 1

In [14]:
#SYNC TILES
dacTarget=-1
adcTarget=-1

In [15]:
if ol.ACTIVE_DAC_TILES > 0:
    ol.xrfdc.mts_dac_config.Tiles = ol.ACTIVE_DAC_TILES # group defined in binary 0b1111
    ol.xrfdc.mts_dac_config.SysRef_Enable = 1
    ol.xrfdc.mts_dac_config.Target_Latency = dacTarget 
    ol.xrfdc.mts_dac()
else:
    ol.xrfdc.mts_dac_config.Tiles = 0x0
    ol.xrfdc.mts_dac_config.SysRef_Enable = 0

In [16]:
if ol.ACTIVE_ADC_TILES > 0:
    ol.xrfdc.mts_adc_config.Tiles = ol.ACTIVE_ADC_TILES
    ol.xrfdc.mts_adc_config.SysRef_Enable = 1
    ol.xrfdc.mts_adc_config.Target_Latency = adcTarget
    ol.xrfdc.mts_adc()
else:
    ol.xrfdc.mts_adc_config.Tiles = 0x0
    ol.xrfdc.mts_adc_config.SysRef_Enable = 0

In [17]:
Pulso=ol.Tx_Concats.Muestras_DDFS_Frec_c_0
Pulso.write(0x0,400) #Indicamos que se genere un pulso de 100 MHz
Mux_cal=ol.Tx_Concats.Mux_Cal_0
Mux_cal.write(0x0,1) # Inyectamos el pulso 

In [18]:
Mux_cal.write(0x0,0)

In [19]:
Mux=ol.Mux_and_Data.Multiplexor_0 

In [20]:
Mux.write(0x0,2) #Datos de entrada al modelo Tx (0: patrón 11001100; 1: patrón 10101010; 2: contador de 8 bits; 3: patrón 00110011)